# Experiments och figures in the report

This notebook contain all tests and visualizations that are used in the report. This file executes the "project.ipynb" in it entirety to run so beware of extrem long execution time (around 5 hours when tested, the parameter tuning is what takes time).

In [ ]:
%run project.ipynb

In [ ]:
import inspect
import os
import pickle
import time
from collections import deque

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

NAV_CURVE_SEEDS  = range(5)        # öppna kartan: seeds för lärkurvan (seed 0 används även i tabellen)
NSTEP_VALUES     = [1, 3, 5, 10]   # n-stegs-jämförelsen på trånga korridoren
NSTEP_SEEDS      = range(5)
NSTEP_EPISODES   = 10000
REWARD_SEEDS     = range(10)       # självisk vs hänsynsfull på trånga korridoren
PICKUP2_EPISODES = 30000           # två agenter som plockar
SWEEP_QINITS     = [0, -30, -100]
SWEEP_SEEDS      = range(3)        # seeds per rad i q_init-svepet
EXTRA_SEEDS      = range(3, 5)      # extra seeds vid q_init=-30, ger 5 seeds totalt för selfish vs cooperative
SNAPSHOT_KEY     = (-30, "selfish", 0)   # vilken körning som ritas som ruteföljd: (q_init, mode, seed)


USE_TUNED     = False            # True: pickup-experimenten tränar varje läge med sin bästa konfiguration
                                # False: funktionernas standardvärden (för att jämföra mot orättade parametrar)
EPS_END_PARAM = "epsilon_end"   # vad parametern för slut-epsilon heter i train_two_pickup (ändra om er version heter något annat)
BEST_REPORTED = {
    "selfish":     dict(alpha=0.1, gamma=0.99, eps_end=0.05, n=1, mean_steps=49.7),
    "cooperative": dict(alpha=0.1, gamma=0.99, eps_end=0.2,  n=3, mean_steps=44.7),
}

COL_SELFISH, COL_COOP = "#0072B2", "#D55E00"       # belöningsläge i lärkurvorna
AGENT_COLORS, AGENT_TEXT = ["#0072B2", "#D11F12"], ["black", "white"]   # agent A och B i rutorna
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10})


# parametrar till train_two_pickup för ett läges bästa konfiguration (tom om USE_TUNED = False).
# episoder, q_init och seed anges av den som anropar, t.ex.
#   train_two_pickup(env, episodes=30000, q_init=-30, seed=0, **tuned_kwargs("selfish"))
def tuned_kwargs(mode):
    if not USE_TUNED:
        return {}
    if EPS_END_PARAM not in inspect.signature(train_two_pickup).parameters:
        raise RuntimeError(f"train_two_pickup har ingen parameter som heter {EPS_END_PARAM!r}. "
                           "Ändra EPS_END_PARAM till vad slut-epsilon heter i er version.")
    cfg = BEST_REPORTED[mode]
    return {"n": cfg["n"], "alpha": cfg["alpha"], "gamma": cfg["gamma"], EPS_END_PARAM: cfg["eps_end"]}

### Gemensamma hjälpfunktioner

* `nav_result` kör en girig episod (ingen slump) och returnerar när varje agent kom i mål och antalet krockar.
* `optimal_joint_steps` räknar ut exakt minsta antal steg tills båda är i mål, med bredden-först över båda agenternas positioner och samma krockregler som `step()`. Det är jämförelsevärdet "optimum" i tabellerna.
* `summarize_runs` gör en tabellrad av flera seeds, och `latex_row` skriver ut den i LaTeX-format (`--` betyder att ingen seed klarade det).

In [ ]:
# kör en girig episod tyst: (steg när A kom i mål, steg när B kom i mål, krockar)
def nav_result(env, Qs, max_steps=50):
    rng = np.random.default_rng(0)
    state = env.reset()
    finished = [None, None]
    crashes = 0
    for t in range(1, max_steps + 1):
        a = [choose_action(Qs[i], state, 0.0, rng) for i in range(2)]
        state, rewards, done = env.step((ACTION_NAMES[a[0]], ACTION_NAMES[a[1]]))
        crashes += sum(r < STEP_REWARD for r in rewards)
        for i in range(2):
            if finished[i] is None and state[i] == env.goals[i]:
                finished[i] = t
        if done:
            break
    return finished[0], finished[1], crashes


# exakt minsta antal steg tills båda är i mål (bredden-först över båda positionerna)
def optimal_joint_steps(env):
    start = env.reset()
    seen = {start}
    queue = deque([(start, 0)])
    result = None
    while queue and result is None:
        state, d = queue.popleft()
        for a0 in ACTION_NAMES:
            for a1 in ACTION_NAMES:
                env.positions = list(state)
                nxt, _, done = env.step((a0, a1))
                if done:
                    result = d + 1
                    break
                if nxt not in seen:
                    seen.add(nxt)
                    queue.append((nxt, d + 1))
            if result is not None:
                break
    env.reset()
    return result


# flera seeds -> (klarade, optimala, medel steg tills båda klara, krockar totalt)
def summarize_runs(results, optimum):
    both = [max(a, b) for a, b, _ in results if a is not None and b is not None]
    mean_steps = float(np.mean(both)) if both else None
    return len(both), sum(t == optimum for t in both), mean_steps, sum(c for _, _, c in results)


def fmt(x):
    if x is None:
        return "--"
    return f"{x:.1f}" if isinstance(x, (float, np.floating)) else str(x)


def latex_row(*cells):
    print(" & ".join(fmt(c) for c in cells) + r" \\")


def make_open_env(reward_mode="selfish"):
    return TwoAgentWarehouse(WarehouseGrid(), starts=[(0, 0), (0, 9)], goals=[(6, 9), (6, 0)],
                             reward_mode=reward_mode)


NARROW_LAYOUT = """
####.####
.........
"""

def make_narrow_env(reward_mode="selfish"):
    return TwoAgentWarehouse(WarehouseGrid(NARROW_LAYOUT), starts=[(1, 0), (1, 8)], goals=[(1, 8), (1, 0)],
                             reward_mode=reward_mode)


open_env, narrow_env = make_open_env(), make_narrow_env()
OPTIMUM = {"open": optimal_joint_steps(open_env), "narrow": optimal_joint_steps(narrow_env)}
print("optimum (steg tills båda i mål, med krockregler):", OPTIMUM)

### Två agenter, navigering (öppna kartan och trånga korridoren)

Fyra körningar med `train()` och standardinställningar om inget annat anges. Öppna kartan tränas med flera seeds för lärkurvan, och seed 0 används som tabellrad.

In [ ]:
t0 = time.time()
nav_curves = []            # steg per episod, en lista per seed (öppna kartan)
nav_table = []

for seed in NAV_CURVE_SEEDS:
    Qs_, steps_ = train(open_env, seed=seed)
    nav_curves.append(steps_)
    if seed == 0:
        nav_table.append(("open", "default", *nav_result(open_env, Qs_)))

for label, kw in [("seed 0", {"seed": 0}),
                  ("seed 1", {"seed": 1}),
                  (r"50\,000 ep., $\alpha = 0.2$", {"episodes": 50000, "alpha": 0.2})]:
    Qs_, _ = train(narrow_env, **kw)
    nav_table.append(("narrow", label, *nav_result(narrow_env, Qs_)))

print(f"klart på {time.time() - t0:.0f} s\n")
print("Map & Setting & A done & B done & Collisions \\\\")
for row in nav_table:
    latex_row(*row)
print("\noptimum (steg tills båda klara):", OPTIMUM)

# with open("results/two_agent_navigation.pkl", "wb") as f:
#     pickle.dump({"table": nav_table, "curves": nav_curves, "optimum": OPTIMUM}, f)

In [ ]:
# lärkurva, öppna kartan: medel över seeds i block om 100 episoder, skuggat = min till max över seeds
block = 100
S = np.array(nav_curves)
S = S[:, : S.shape[1] // block * block]
m = S.reshape(S.shape[0], -1, block).mean(axis=2)
x = np.arange(m.shape[1]) * block + block / 2

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(x, m.mean(axis=0), color=COL_SELFISH, lw=1.8, label=f"mean of {S.shape[0]} seeds")
ax.fill_between(x, m.min(axis=0), m.max(axis=0), color=COL_SELFISH, alpha=0.15, lw=0, label="min to max")
ax.axhline(OPTIMUM["open"], color="0.3", lw=1, ls=":", label=f"optimum ({OPTIMUM['open']} steps)")
ax.set_xlabel("episode")
ax.set_ylabel(f"steps per episode (mean of {block} episodes)")
ax.set_ylim(bottom=0)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("figures/two_agent_navigation_learning_curve.png", dpi=200)
plt.show()

### $n$-steg mot ett steg på trånga korridoren

`train_nstep` för varje `n`, flera seeds. "finished" är hur många seeds där båda kom i mål, "optimal" hur många av dem som tog exakt optimumsteg, "mean steps" är medel över de som klarade det, och krockarna är summan över seeds.

In [ ]:
t0 = time.time()
nstep_runs = {}
for n in NSTEP_VALUES:
    nstep_runs[n] = []
    for seed in NSTEP_SEEDS:
        Qs_, _ = train_nstep(narrow_env, n=n, episodes=NSTEP_EPISODES, seed=seed)
        nstep_runs[n].append(nav_result(narrow_env, Qs_))
    print(f"n={n} klar ({time.time() - t0:.0f} s)", flush=True)

n_seeds = len(NSTEP_SEEDS)
print(f"\noptimum = {OPTIMUM['narrow']} steg. Rader (finished/optimal som antal av {n_seeds} seeds):")
print(r"$n$ & finished & optimal & mean steps & collisions \\")
for n in NSTEP_VALUES:
    finished, optimal, mean_steps, crashes = summarize_runs(nstep_runs[n], OPTIMUM["narrow"])
    latex_row(n, f"{finished}/{n_seeds}", f"{optimal}/{n_seeds}", mean_steps, crashes)

# with open("results/narrow_nstep.pkl", "wb") as f:
#     pickle.dump(nstep_runs, f)

### Självisk mot hänsynsfull belöning på trånga korridoren

`train` (ett steg) med `reward_mode="selfish"` respektive `"cooperative"`, samma kolumner som ovan.

In [ ]:
t0 = time.time()
reward_runs = {}
for mode in ["selfish", "cooperative"]:
    env_mode = make_narrow_env(mode)
    reward_runs[mode] = [nav_result(env_mode, train(env_mode, seed=seed)[0]) for seed in REWARD_SEEDS]
    print(f"{mode} klar ({time.time() - t0:.0f} s)", flush=True)

n_seeds = len(REWARD_SEEDS)
print(f"\noptimum = {OPTIMUM['narrow']} steg. Rader:")
print(r"Reward mode & finished & optimal & mean steps until both done & collisions \\")
for mode in ["selfish", "cooperative"]:
    finished, optimal, mean_steps, crashes = summarize_runs(reward_runs[mode], OPTIMUM["narrow"])
    latex_row(mode, f"{finished}/{n_seeds}", f"{optimal}/{n_seeds}", mean_steps, crashes)

print("\nPer seed (A klar, B klar, krockar):")
for mode in ["selfish", "cooperative"]:
    print(f"  {mode:12s}", reward_runs[mode])

# with open("results/narrow_reward_modes.pkl", "wb") as f:
#     pickle.dump(reward_runs, f)

### Två agenter som plockar: `q_init`-svep och självisk mot hänsynsfull

Kartan och varorna är samma som i steg 7. Varje rad i tabellen är medel över `SWEEP_SEEDS` seeds. Gapet är antal steg utöver den undre gränsen (varje agent ensam i lagret). "finished" räknar de seeds där båda agenterna blev klara (allt levererat och tillbaka på sin start) inom 200 steg, och A/B done är medel över just de seeds.

Varje belöningsläge tränas med sina bästa parametrar från steg 9: `alpha`, `gamma`, slut-epsilon och `n` enligt `BEST_REPORTED` i inställningscellen, via `tuned_kwargs(mode)`. Sätt `USE_TUNED = False` för att i stället använda funktionernas standardvärden. Parametrarna hittades med 10000 episoder, medan `PICKUP2_EPISODES` här är 30000, och `q_init` varierar i svepet.

Obs: `PICKUP2_EPISODES` är 30000 och en träning tar minuter. Med 18 träningar i svepet plus 4 extra är det lång tid, så testa först med få episoder och få seeds i inställningscellen.

In [ ]:
pick_grid = WarehouseGrid(make_layout(n_shelf_rows=4, shelves_per_row=2, shelf_width=3, shelf_depth=1))
pick_grid.place_items_randomly([0, 1, 2, 3], seed=1)

def make_pickup2_env(mode):
    return TwoAgentPickupWarehouse(pick_grid, items_per_agent=[[0, 1], [2, 3]],
                                   starts=[(0, 0), (0, 8)], reward_mode=mode)

BOUNDS = lower_bound_steps(make_pickup2_env("selfish"))
print(f"undre gräns, ensam i lagret: A {BOUNDS[0]} steg, B {BOUNDS[1]} steg")
print("parametrar per läge:", {m: tuned_kwargs(m) for m in ["selfish", "cooperative"]})

pickup2_runs = {}     # (q_init, mode) -> [(a_done, b_done, crashes, levererat), ...] per seed
pickup2_curves = {}   # (q_init, mode) -> [steg per episod, ...] per seed
snapshot_Qs = None

def run_pickup2(q_init, mode, seed):
    global snapshot_Qs
    env_ = make_pickup2_env(mode)
    t0 = time.time()
    Qs_, steps_ = train_two_pickup(env_, episodes=PICKUP2_EPISODES, q_init=q_init, seed=seed,
                                   **tuned_kwargs(mode))
    result = run_greedy_two_pickup(env_, Qs_)
    pickup2_runs.setdefault((q_init, mode), []).append(result)
    pickup2_curves.setdefault((q_init, mode), []).append(steps_)
    if (q_init, mode, seed) == SNAPSHOT_KEY:
        snapshot_Qs = Qs_
    print(f"q_init {q_init:5d}, {mode:12s}, seed {seed}: A {result[0]}, B {result[1]}, "
          f"krockar {result[2]}, levererat {result[3]}  ({time.time() - t0:.0f} s)", flush=True)

for q_init in SWEEP_QINITS:
    for mode in ["selfish", "cooperative"]:
        for seed in SWEEP_SEEDS:
            run_pickup2(q_init, mode, seed)

# with open("results/two_agent_pickup.pkl", "wb") as f:
#     pickle.dump({"runs": pickup2_runs, "curves": pickup2_curves, "bounds": BOUNDS,
#                  "params": {m: tuned_kwargs(m) for m in ["selfish", "cooperative"]}}, f)

In [ ]:
# tabellen: medel över seeds, gap = steg utöver ensam-gränsen
def pickup2_row(runs, bounds):
    ok = [r for r in runs if r[0] is not None and r[1] is not None]
    def done_gap(i):
        if not ok:
            return "--"
        m = float(np.mean([r[i] for r in ok]))
        return f"{m:.1f} ({m - bounds[i]:+.1f})"
    return f"{len(ok)}/{len(runs)}", done_gap(0), done_gap(1), float(np.mean([r[2] for r in runs]))

print(f"Rader (medel över {len(SWEEP_SEEDS)} seeds, gräns A {BOUNDS[0]} / B {BOUNDS[1]}):")
print(r"Reward mode & $Q$ init & finished & A done (gap) & B done (gap) & Collisions \\")
for q_init in SWEEP_QINITS:
    for mode in ["selfish", "cooperative"]:
        runs = pickup2_runs[(q_init, mode)][: len(SWEEP_SEEDS)]
        latex_row(mode, f"${q_init}$", *pickup2_row(runs, BOUNDS))

Extra seeds vid `q_init = -30` så att självisk mot hänsynsfull har fem seeds (de tre första återanvänds från svepet). Samma parametrar per läge som i svepet.

In [ ]:
for mode in ["selfish", "cooperative"]:
    for seed in EXTRA_SEEDS:
        run_pickup2(-30, mode, seed)

print(f"\nq_init = -30, {len(SWEEP_SEEDS) + len(EXTRA_SEEDS)} seeds:")
for mode in ["selfish", "cooperative"]:
    runs = pickup2_runs[(-30, mode)]
    finished, a_done, b_done, crashes = pickup2_row(runs, BOUNDS)
    print(f"  {mode:12s} finished {finished}, A done {a_done}, B done {b_done}, krockar (medel) {crashes:.1f}")
    print(f"               per seed (A, B, krockar): {[r[:3] for r in runs]}")

# with open("results/two_agent_pickup.pkl", "wb") as f:
#     pickle.dump({"runs": pickup2_runs, "curves": pickup2_curves, "bounds": BOUNDS,
#                  "params": {m: tuned_kwargs(m) for m in ["selfish", "cooperative"]}}, f)

### Figurer: lärkurvor och ruteföljd för två plockande agenter

Lärkurvorna är medel över seeds, en ruta per `q_init`, med självisk heldragen och hänsynsfull streckad. Ruteföljden visar den giriga körningen för `SNAPSHOT_KEY` (självisk, `q_init = -30`, seed 0), en ruta vid varje plock, leverans och `WAIT` (flera `WAIT` i rad räknas som en).

In [ ]:
# lärkurvor: medel steg per episod i block om 1000, skuggat = min till max över seeds
fig, axes = plt.subplots(1, len(SWEEP_QINITS), figsize=(9, 3.4), sharey=True, squeeze=False)
axes = axes.ravel()
block = 1000
for ax, q_init in zip(axes, SWEEP_QINITS):
    for mode, color, ls in [("selfish", COL_SELFISH, "-"), ("cooperative", COL_COOP, "--")]:
        S = np.array(pickup2_curves[(q_init, mode)][: len(SWEEP_SEEDS)])
        S = S[:, : S.shape[1] // block * block]
        m = S.reshape(S.shape[0], -1, block).mean(axis=2)
        x = np.arange(m.shape[1]) * block + block / 2
        ax.plot(x, m.mean(axis=0), color=color, ls=ls, lw=1.6, label=mode)
        ax.fill_between(x, m.min(axis=0), m.max(axis=0), color=color, alpha=0.12, lw=0)
    ax.set_title(f"Q init = {q_init}", fontsize=10)
    ax.set_xlabel("episode")
axes[0].set_ylabel(f"steps per episode (mean of {block})")
axes[0].set_ylim(bottom=0)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)   # utanför rutorna, täcker inga kurvor
fig.tight_layout(rect=(0, 0, 1, 0.92))
fig.savefig("figures/two_agent_pickup_learning_curve.png", dpi=200)
plt.show()

In [ ]:
# girig körning som sparar varje steg (samma slumpanvändning som run_greedy_two_pickup)
def rollout_two_pickup(env, Qs, max_steps=200):
    rng = np.random.default_rng(0)
    env.reset()
    frames = [dict(t=0, pos=list(env.pos), carrying=list(env.carrying), mask=list(env.mask), events=[], done=False)]
    waiting = [False, False]
    for t in range(1, max_steps + 1):
        was_finished = [env.finished(0), env.finished(1)]
        a = [choose_action_pick(Qs[i], env.obs(i), 0.0, rng) for i in range(2)]
        mask_before, carry_before = list(env.mask), list(env.carrying)
        _, _, done = env.step((ACTIONS_PICK[a[0]], ACTIONS_PICK[a[1]]))
        events = []
        for i in range(2):
            if was_finished[i]:
                waiting[i] = False
                continue
            if env.mask[i] != mask_before[i]:
                events.append((i, "picks"))
            elif carry_before[i] and not env.carrying[i]:
                events.append((i, "delivers"))
            if ACTIONS_PICK[a[i]] == "WAIT":
                if not waiting[i]:            # bara första WAIT i en följd
                    events.append((i, "waits"))
                waiting[i] = True
            else:
                waiting[i] = False
        frames.append(dict(t=t, pos=list(env.pos), carrying=list(env.carrying), mask=list(env.mask),
                           events=events, done=done))
        if done:
            break
    return frames


def snapshot_title(frame, is_first):
    if is_first:
        return "start"
    text = ", ".join(f"{'AB'[i]} {verb}" for i, verb in frame["events"])
    if frame["done"]:
        text = (text + ", " if text else "") + "done"
    return f"t={frame['t']}: {text}" if text else f"t={frame['t']}"


def draw_snapshot(ax, env, frames, k, k_prev, title):
    frame = frames[k]
    ax.set_xlim(-0.5, env.grid.n_cols - 0.5)
    ax.set_ylim(env.grid.n_rows - 0.5, -0.5)      # rad 0 överst
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True); s.set_color("0.7")
    cell = lambda rc, **kw: ax.add_patch(Rectangle((rc[1] - 0.5, rc[0] - 0.5), 1, 1, **kw))
    for rc in env.grid.shelf_cells:                 # hyllor
        cell(rc, facecolor="#dddddd", edgecolor="white", linewidth=1)
    for i in range(2):                              # varor: färgad ruta med siffra, kryss när plockad
        for b, item in enumerate(env.items[i]):
            rc = env.grid.item_to_shelf[item]
            if frame["mask"][i] & (1 << b):
                ax.text(rc[1], rc[0], "x", ha="center", va="center", fontsize=9, color="0.45")
            else:
                cell(rc, facecolor=AGENT_COLORS[i], alpha=0.45, edgecolor="white", linewidth=1)
                ax.text(rc[1], rc[0], str(item), ha="center", va="center", fontsize=8, color="black")
    for i in range(2):                              # hem (start = mål), streckad ruta
        cell(env.starts[i], fill=False, edgecolor=AGENT_COLORS[i], linestyle="--", linewidth=1.5)
    cell(env.dropoff, facecolor="white", edgecolor="black", linewidth=1.5)
    ax.text(env.dropoff[1], env.dropoff[0], "D", ha="center", va="center", fontsize=8, fontweight="bold")
    for i in range(2):                              # väg sedan förra rutan, lite förskjuten så att de syns båda
        off = (-0.12, 0.12)[i]
        xs = [frames[j]["pos"][i][1] + off for j in range(k_prev, k + 1)]
        ys = [frames[j]["pos"][i][0] + off for j in range(k_prev, k + 1)]
        ax.plot(xs, ys, color=AGENT_COLORS[i], lw=2.2, alpha=0.8, solid_capstyle="round", zorder=3)
    for i in range(2):                              # agenterna, svart ring när de bär en vara
        r, c = frame["pos"][i]
        ax.scatter([c], [r], s=230, facecolor=AGENT_COLORS[i],
                   edgecolor="black" if frame["carrying"][i] else AGENT_COLORS[i], linewidth=2.5, zorder=5)
        ax.text(c, r, "AB"[i], ha="center", va="center", fontsize=8, fontweight="bold",
                color=AGENT_TEXT[i], zorder=6)
    ax.set_title(title, fontsize=9)


def plot_snapshots(env, frames, path, ncols=4, max_panels=24):
    ks = [0] + [k for k in range(1, len(frames)) if frames[k]["events"]]
    if len(frames) - 1 not in ks:
        ks.append(len(frames) - 1)                  # sista rutan
    if len(ks) > max_panels:
        print(f"Obs: {len(ks)} rutor, ritar bara de första {max_panels}")
        ks = ks[:max_panels]
    nrows = -(-len(ks) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 3.2 * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for p, k in enumerate(ks):
        ax = axes.ravel()[p]
        ax.axis("on")
        draw_snapshot(ax, env, frames, k, ks[p - 1] if p > 0 else 0, snapshot_title(frames[k], k == 0))
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.show()
    return ks


snap_env = make_pickup2_env(SNAPSHOT_KEY[1])
frames = rollout_two_pickup(snap_env, snapshot_Qs)
print(f"{len(frames) - 1} steg, klar: {frames[-1]['done']}")
shown = plot_snapshots(snap_env, frames, "figures/two_agent_pickup_snapshots.png")
print("rutor:", [snapshot_title(frames[k], k == 0) for k in shown])